# Lot 패턴 분류 — threshold 탐색

다양한 위험/안전 threshold에서 lot별 웨이퍼맵과 패턴(normal/random/edge/**center**)을 확인한다.
`regen_pattern.py`와 동일한 분류 로직·집계(좌표별 max, 원본 lot 1~28).

**핵심**: center는 `center_avg > edge_avg × RATIO` 조건이 먼저 만족돼야 함 → 이건 **threshold와 무관**(공간 집중도).
그래서 (1) 먼저 center가 가능한 lot이 있는지 진단하고 (2) threshold sweep으로 분포를 보고 (3) 슬라이더로 맵을 본다.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

P = Path('C:/Users/Dell3571/Desktop/dashboard_v3/data/processed')
wm = pd.read_csv(P / 'wafer_map.csv')
wm['pred'] = pd.to_numeric(wm['pred'], errors='coerce')
ws = json.load(open(P / 'wafer_scale.json'))
BASE_THRESHOLD = ws['threshold']  # 현재 대시보드 기준 = die pred P90
print('기본 threshold (P90):', BASE_THRESHOLD)

# 대상 lot: 원본 1~28 (regen_pattern.py와 동일)
target = [int(l) for l in wm['run_id'].unique() if 1 <= int(l) <= 28]

# lot별 좌표 max 집계 (계층탐색 lot맵 = lotAccumDies와 동일)
lot_dies = {}
for lot, g in wm.groupby('run_id'):
    if int(lot) not in target:
        continue
    coord = g.groupby(['die_x', 'die_y'], as_index=False)['pred'].max().dropna(subset=['pred'])
    lot_dies[int(lot)] = coord[['die_x', 'die_y', 'pred']].values
print('lot 수:', len(lot_dies))

allp = wm['pred'].dropna()
print('die pred 분위수:', {f'P{q}': round(float(allp.quantile(q/100)), 6) for q in [50, 75, 90, 95, 99]})

In [ ]:
# 분류 파라미터 (여기서 조절 가능)
RATIO    = 1.6    # 편중 배수 (center/edge 판정) — 낮추면 center가 더 잘 잡힘
R_CENTER = 0.45   # 중심 반경 (정규화)
R_EDGE   = 0.75   # 외곽 반경 (정규화)
HIGH_RATIO = 0.10 # normal vs random 경계 (위험 die 비율)

def lot_stats(dies):
    """center_avg, edge_avg 반환 (threshold 무관). 영역 비면 None."""
    xs, ys, ps = dies[:, 0], dies[:, 1], dies[:, 2]
    cx = (xs.max() + xs.min()) / 2
    cy = (ys.max() + ys.min()) / 2
    r = np.hypot(xs - cx, (ys - cy) * 2.5)
    rmax = r.max()
    if rmax == 0:
        return None
    rn = r / rmax
    cm = rn < R_CENTER
    em = rn > R_EDGE
    if cm.sum() == 0 or em.sum() == 0:
        return None
    return ps[cm].mean(), ps[em].mean()

def classify(dies, threshold):
    st = lot_stats(dies)
    if st is None:
        return 'normal'
    center_avg, edge_avg = st
    high_ratio = (dies[:, 2] > threshold).mean()
    if edge_avg > center_avg * RATIO and edge_avg > threshold * 0.3:
        return 'edge'
    if center_avg > edge_avg * RATIO and center_avg > threshold * 0.3:
        return 'center'
    if high_ratio < HIGH_RATIO:
        return 'normal'
    return 'random'

## 1) center가 가능한 lot 진단 (threshold 무관)

`center_avg > edge_avg × RATIO`를 만족하는 lot이 하나도 없으면 **threshold를 아무리 바꿔도 center는 안 나온다.**
이 경우 위 셀의 `RATIO`(예: 1.3)나 `R_CENTER`/`R_EDGE`를 조절해야 함.

In [ ]:
recs = []
for lot, dies in sorted(lot_dies.items()):
    st = lot_stats(dies)
    if st is None:
        recs.append({'lot': lot, 'center_avg': np.nan, 'edge_avg': np.nan, 'c/e': np.nan, 'center가능': False})
        continue
    c, e = st
    recs.append({'lot': lot, 'center_avg': round(c, 6), 'edge_avg': round(e, 6),
                 'c/e': round(c / e, 2) if e else np.nan, 'center가능': bool(c > e * RATIO)})
diag = pd.DataFrame(recs)
print(f'center 조건(center_avg > edge_avg×{RATIO}) 만족 lot: {int(diag["center가능"].sum())}개')
print(f'c/e 비율 최댓값: {diag["c/e"].max():.2f} (lot {int(diag.loc[diag["c/e"].idxmax(), "lot"])})')
diag.sort_values('c/e', ascending=False)

## 2) threshold sweep — 패턴 분포

die pred 분위수 전체를 훑으며 각 threshold에서 lot 패턴 개수를 센다. center가 1개라도 나오는 threshold가 있는지 확인.

In [ ]:
cands = sorted(set([float(allp.quantile(q)) for q in np.arange(0.05, 1.00, 0.05)] + [float(BASE_THRESHOLD)]))
sweep = []
for t in cands:
    cnt = {'normal': 0, 'random': 0, 'edge': 0, 'center': 0}
    for lot, dies in lot_dies.items():
        cnt[classify(dies, t)] += 1
    sweep.append({'threshold': round(t, 6), **cnt})
sweep_df = pd.DataFrame(sweep)
has_center = sweep_df[sweep_df['center'] > 0]
print('center가 나오는 threshold:', '없음' if has_center.empty else f'{len(has_center)}개 구간')
if not has_center.empty:
    display(has_center)
sweep_df

## 3) lot별 웨이퍼맵 시각화

특정 threshold에서 전체 lot 맵을 한 화면에. 빨강 = die pred > threshold(위험), 파랑 = 정상. 제목에 분류 결과(center면 빨간 글씨).

In [ ]:
def draw_lot_maps(threshold, ncols=7):
    lots = sorted(lot_dies.keys())
    nrows = int(np.ceil(len(lots) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.0, nrows * 2.1))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis('off')
    counts = {}
    for ax, lot in zip(axes, lots):
        dies = lot_dies[lot]
        pat = classify(dies, threshold)
        counts[pat] = counts.get(pat, 0) + 1
        risk = dies[:, 2] > threshold
        ax.scatter(dies[~risk, 0], dies[~risk, 1], c='#cfe3f7', s=7, marker='s')
        ax.scatter(dies[risk, 0], dies[risk, 1], c='#dc2626', s=7, marker='s')
        ax.set_title(f'Lot{lot} | {pat}', fontsize=8,
                     color='#dc2626' if pat == 'center' else ('#ea580c' if pat == 'edge' else '#111'))
        ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
    plt.suptitle(f'threshold = {threshold:.6f}   →   {counts}', fontsize=11)
    plt.tight_layout(); plt.show()

draw_lot_maps(BASE_THRESHOLD)

## 4) threshold 슬라이더 (인터랙티브)

슬라이더를 움직이며 패턴 변화를 관찰. ipywidgets 미설치 시 `draw_lot_maps(0.0005)`처럼 직접 호출.

In [ ]:
try:
    from ipywidgets import interact, FloatSlider
    pmin, pmax = float(allp.quantile(0.30)), float(allp.quantile(0.99))
    interact(lambda threshold: draw_lot_maps(threshold),
             threshold=FloatSlider(min=pmin, max=pmax, step=(pmax - pmin) / 60,
                                   value=float(BASE_THRESHOLD), readout_format='.6f'))
except Exception as e:
    print('ipywidgets 미설치 — draw_lot_maps(원하는_threshold)로 직접 확인하세요. 예: draw_lot_maps(0.0005)')
    print(e)